In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [19]:
# !pip install pandas numpy matplotlib seaborn scikit-learn xgboost joblib

In [5]:
df = pd.read_csv('../data/t20i_Matches_Data.csv')

In [6]:
df.head()

,T20I Match No,Match ID,Match Name,Series ID,Series Name,Match Date,Match Format,Team1 ID,Team1 Name,Team1 Captain,...,Umpire 2,Match Referee,Toss Winner,Toss Winner Choice,Match Winner,Match Result Text,MOM Player,Team1 Playing 11,Team2 Playing 11,Debut Players
0,52,291356,Australia Vs India Only T20I,291355,India tour of Australia - 2007 (2007/08),2008-02-01,T20,6,India,7593,...,SJA Taufel,JJ Crowe,India,bat,Australia,Australia won by 9 wickets (with 52 balls rema...,8876.0,"['7773', '7781', '8813', '8742', '48405', '759...","['4176', '8876', '6253', '6256', '4382', '1198...","['11984', '49327', '48319']"
1,54,300436,New Zealand Vs England 2Nd T20I,300418,England tour of New Zealand - 2008 (2007/08),2008-02-07,T20,1,England,2314,...,GAV Baxter,AG Hurst,England,bat,England,England won by 50 runs,2314.0,"['11556', '44660', '8107', '7822', '2314', '63...","['44946', '10384', '44930', '10381', '9570', '...","['47488', '10325']"
2,65,361531,Netherlands Vs Scotland 2Nd Semi Final,353665,"ICC World Twenty20 Qualifier Bermuda, Canada, ...",2008-08-04,T20,30,Scotland,45548,...,PK Baldwin,BC Broad,Netherlands,bowl,Netherlands,Netherlands won by 5 wickets (with 12 balls re...,45358.0,"['45548', '46048', '46142', '8221', '4334', '4...","['10323', '48655', '6362', '49443', '45358', '...",[]
3,66,354459,Kenya Vs Scotland 3Rd Place Playoff,353665,"ICC World Twenty20 Qualifier Bermuda, Canada, ...",2008-08-04,T20,26,Kenya,2265,...,PK Baldwin,BC Broad,Kenya,bat,Scotland,Scotland won by 9 wickets (with 11 balls remai...,45548.0,"['10364', '2264', '49383', '2265', '2268', '50...","['45548', '46048', '46142', '8221', '4334', '4...","['50293', '50293']"
4,69,361653,Sri Lanka Vs Zimbabwe 1St Match,361644,T20 Canada in Canada - 2008 (2008/09),2008-10-10,T20,9,Zimbabwe,45326,...,MR Benson,JJ Crowe,Sri Lanka,bowl,Sri Lanka,Sri Lanka won by 5 wickets (with 6 balls remai...,50377.0,"['10639', '10423', '47619', '10421', '21364', ...","['48468', '7419', '15273', '8195', '6315', '48...","['50377', '47210', '12209', '15273', '48468', ..."


In [20]:
df.columns

Index(['T20I Match No', 'Match ID', 'Match Name', 'Series ID', 'Series Name',
       'Match Date', 'Match Format', 'Team1 ID', 'Team1 Name', 'Team1 Captain',
       'Team1 Runs Scored', 'Team1 Wickets Fell', 'Team1 Extras Rec',
       'Team2 ID', 'Team2 Name', 'Team2 Captain', 'Team2 Runs Scored',
       'Team2 Wickets Fell', 'Team2 Extras Rec', 'Match Venue (Stadium)',
       'Match Venue (City)', 'Match Venue (Country)', 'Umpire 1', 'Umpire 2',
       'Match Referee', 'Toss Winner', 'Toss Winner Choice', 'Match Winner',
       'Match Result Text', 'MOM Player', 'Team1 Playing 11',
       'Team2 Playing 11', 'Debut Players'],
      dtype='str')

In [21]:
selected_columns = [
    'Match Date',
    'Team1 Name',
    'Team2 Name',
    'Match Venue (Stadium)',
    'Toss Winner',
    'Toss Winner Choice',
    'Match Winner'
]

df_filtered = df[selected_columns].copy()

In [22]:
df_filtered

,Match Date,Team1 Name,Team2 Name,Match Venue (Stadium),Toss Winner,Toss Winner Choice,Match Winner
0,2008-02-01,India,Australia,Melbourne Cricket Ground,India,bat,Australia
1,2008-02-07,England,New Zealand,Jade Stadium,England,bat,England
2,2008-08-04,Scotland,Netherlands,Civil Service Cricket Club,Netherlands,bowl,Netherlands
3,2008-08-04,Kenya,Scotland,Civil Service Cricket Club,Kenya,bat,Scotland
4,2008-10-10,Zimbabwe,Sri Lanka,Maple Leaf North-West Ground,Sri Lanka,bowl,Sri Lanka
...,...,...,...,...,...,...,...
2587,2024-05-05,Thailand,Indonesia,Udayana Cricket Ground,Indonesia,bowl,Thailand
2588,2024-05-05,Zimbabwe,Bangladesh,Zahur Ahmed Chowdhury Stadium,Bangladesh,bowl,Bangladesh
2589,2024-05-06,Indonesia,Thailand,Udayana Cricket Ground,Indonesia,bat,Thailand
2590,2024-05-07,Japan,Mongolia,Sano International Cricket Ground,Japan,bat,Japan


In [24]:
df_filtered = df_filtered.rename(columns={
    'Match Date': 'date',
    'Team1 Name': 'team1',
    'Team2 Name': 'team2',
    'Match Venue (Stadium)': 'venue',
    'Toss Winner': 'toss_winner',
    'Toss Winner Choice': 'toss_decision',
    'Match Winner': 'winner'
})

df_filtered.head()

,date,team1,team2,venue,toss_winner,toss_decision,winner
0,2008-02-01,India,Australia,Melbourne Cricket Ground,India,bat,Australia
1,2008-02-07,England,New Zealand,Jade Stadium,England,bat,England
2,2008-08-04,Scotland,Netherlands,Civil Service Cricket Club,Netherlands,bowl,Netherlands
3,2008-08-04,Kenya,Scotland,Civil Service Cricket Club,Kenya,bat,Scotland
4,2008-10-10,Zimbabwe,Sri Lanka,Maple Leaf North-West Ground,Sri Lanka,bowl,Sri Lanka


In [34]:
df_filtered.isnull().sum()

date             11
team1             0
team2             0
venue             0
toss_winner       1
toss_decision    10
winner           98
dtype: int64

In [ ]:
null_percentage = (df_filtered.isnull().sum() / len(df_filtered)) * 100
print(null_percentage)

date             0.424383
team1            0.000000
team2            0.000000
venue            0.000000
toss_winner      0.038580
toss_decision    0.385802
winner           3.780864
dtype: float64


In [36]:
df_cleaned = df_filtered.dropna().reset_index(drop=True)

In [ ]:
df_cleaned = df_cleaned[df_cleaned['winner'].isin(df_cleaned['team1']) | df_cleaned['winner'].isin(df_cleaned['team2'])].reset_index(drop=True)

print(f"Original shape: {df_filtered.shape}")
print(f"Cleaned shape:  {df_cleaned.shape}")
print("\nMissing values left:")
print(df_cleaned.isnull().sum())

Original shape: (2592, 7)
Cleaned shape:  (2483, 7)

Missing values left:
date             0
team1            0
team2            0
venue            0
toss_winner      0
toss_decision    0
winner           0
dtype: int64


In [39]:
df_cleaned

,date,team1,team2,venue,toss_winner,toss_decision,winner
0,2008-02-01,India,Australia,Melbourne Cricket Ground,India,bat,Australia
1,2008-02-07,England,New Zealand,Jade Stadium,England,bat,England
2,2008-08-04,Scotland,Netherlands,Civil Service Cricket Club,Netherlands,bowl,Netherlands
3,2008-08-04,Kenya,Scotland,Civil Service Cricket Club,Kenya,bat,Scotland
4,2008-10-10,Zimbabwe,Sri Lanka,Maple Leaf North-West Ground,Sri Lanka,bowl,Sri Lanka
...,...,...,...,...,...,...,...
2478,2024-05-05,Thailand,Indonesia,Udayana Cricket Ground,Indonesia,bowl,Thailand
2479,2024-05-05,Zimbabwe,Bangladesh,Zahur Ahmed Chowdhury Stadium,Bangladesh,bowl,Bangladesh
2480,2024-05-06,Indonesia,Thailand,Udayana Cricket Ground,Indonesia,bat,Thailand
2481,2024-05-07,Japan,Mongolia,Sano International Cricket Ground,Japan,bat,Japan


In [40]:
df_cleaned['date'] = pd.to_datetime(df_cleaned['date'])
df_cleaned['match_month'] = df_cleaned['date'].dt.month
df_cleaned['is_weekend'] = df_cleaned['date'].dt.dayofweek.isin([5, 6]).astype(int)

df_cleaned = df_cleaned.sort_values('date').reset_index(drop=True)

In [41]:
df_cleaned

,date,team1,team2,venue,toss_winner,toss_decision,winner,match_month,is_weekend
0,2006-06-15,Sri Lanka,England,The Rose Bowl,Sri Lanka,bat,Sri Lanka,6,0
1,2006-08-28,England,Pakistan,County Ground,England,bat,Pakistan,8,0
2,2006-11-28,Bangladesh,Zimbabwe,Khulna Divisional Stadium,Zimbabwe,bowl,Bangladesh,11,0
3,2006-12-01,South Africa,India,New Wanderers Stadium,South Africa,bat,India,12,0
4,2006-12-22,New Zealand,Sri Lanka,Westpac Stadium,New Zealand,bat,Sri Lanka,12,0
...,...,...,...,...,...,...,...,...,...
2478,2024-05-05,Thailand,Indonesia,Udayana Cricket Ground,Indonesia,bowl,Thailand,5,1
2479,2024-05-05,Zimbabwe,Bangladesh,Zahur Ahmed Chowdhury Stadium,Bangladesh,bowl,Bangladesh,5,1
2480,2024-05-06,Indonesia,Thailand,Udayana Cricket Ground,Indonesia,bat,Thailand,5,0
2481,2024-05-07,Japan,Mongolia,Sano International Cricket Ground,Japan,bat,Japan,5,0


In [43]:
def calculate_team_form(data, window=5):
    team_history = {}
    team1_form, team2_form = [], []
    
    for _, row in data.iterrows():
        t1, t2, w = row['team1'], row['team2'], row['winner']
        
        # Team 1 form
        h1 = team_history.get(t1, [])
        team1_form.append(np.mean(h1[-window:]) if len(h1) > 0 else 0.5)
        
        # Team 2 form
        h2 = team_history.get(t2, [])
        team2_form.append(np.mean(h2[-window:]) if len(h2) > 0 else 0.5)
        
        # History update (1 = win, 0 = loss)
        team_history.setdefault(t1, []).append(1 if w == t1 else 0)
        team_history.setdefault(t2, []).append(1 if w == t2 else 0)
        
    data['team1_form'] = team1_form
    data['team2_form'] = team2_form
    return data

df_cleaned = calculate_team_form(df_cleaned)

In [44]:
df_cleaned

,date,team1,team2,venue,toss_winner,toss_decision,winner,match_month,is_weekend,team1_form,team2_form
0,2006-06-15,Sri Lanka,England,The Rose Bowl,Sri Lanka,bat,Sri Lanka,6,0,0.5,0.5
1,2006-08-28,England,Pakistan,County Ground,England,bat,Pakistan,8,0,0.0,0.5
2,2006-11-28,Bangladesh,Zimbabwe,Khulna Divisional Stadium,Zimbabwe,bowl,Bangladesh,11,0,0.5,0.5
3,2006-12-01,South Africa,India,New Wanderers Stadium,South Africa,bat,India,12,0,0.5,0.5
4,2006-12-22,New Zealand,Sri Lanka,Westpac Stadium,New Zealand,bat,Sri Lanka,12,0,0.5,1.0
...,...,...,...,...,...,...,...,...,...,...,...
2478,2024-05-05,Thailand,Indonesia,Udayana Cricket Ground,Indonesia,bowl,Thailand,5,1,0.4,0.4
2479,2024-05-05,Zimbabwe,Bangladesh,Zahur Ahmed Chowdhury Stadium,Bangladesh,bowl,Bangladesh,5,1,0.2,0.4
2480,2024-05-06,Indonesia,Thailand,Udayana Cricket Ground,Indonesia,bat,Thailand,5,0,0.4,0.4
2481,2024-05-07,Japan,Mongolia,Sano International Cricket Ground,Japan,bat,Japan,5,0,0.4,0.0


In [45]:
# Feature interaction: Toss winner matches Team 1
df_cleaned['toss_winner_is_team1'] = (df_cleaned['toss_winner'] == df_cleaned['team1']).astype(int)

# Feature interaction: Toss decision + Toss winner
df_cleaned['team1_batting_first'] = np.where(
    (df_cleaned['toss_winner_is_team1'] == 1) & (df_cleaned['toss_decision'] == 'bat'), 1,
    np.where((df_cleaned['toss_winner_is_team1'] == 0) & (df_cleaned['toss_decision'] == 'field'), 1, 0)
)

In [46]:
df_cleaned['target'] = (df_cleaned['winner'] == df_cleaned['team1']).astype(int)

In [47]:
feature_cols = [
    'team1', 'team2', 'venue', 'toss_decision',
    'team1_form', 'team2_form', 'toss_winner_is_team1', 
    'team1_batting_first', 'match_month', 'is_weekend'
]

X = df_cleaned[feature_cols]
y = df_cleaned['target']

# 80/20 Time-based split
split_idx = int(len(df_cleaned) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

In [48]:
categorical_features = ['team1', 'team2', 'venue', 'toss_decision']
numerical_features = ['team1_form', 'team2_form', 'match_month']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', StandardScaler(), numerical_features)
    ],
    remainder='passthrough'
)

In [49]:
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42))
])

# Train model
model_pipeline.fit(X_train, y_train)

# Evaluation
y_pred = model_pipeline.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

Accuracy: 0.5835
              precision    recall  f1-score   support

           0       0.65      0.50      0.56       269
           1       0.54      0.68      0.60       228

    accuracy                           0.58       497
   macro avg       0.59      0.59      0.58       497
weighted avg       0.60      0.58      0.58       497



In [50]:
joblib.dump(model_pipeline, 'cricket_pipeline.pkl')
print("Model pipeline successfully exported to cricket_pipeline.pkl")

Model pipeline successfully exported to cricket_pipeline.pkl
